In [1]:
# libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv("../data/processed/cleaned_orders.csv")

In [3]:
df.head()

,Delivery_person_Age,Delivery_person_Ratings,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,delivery_time_mins,weather_clean,distance_km,order_hour,pickup_hour,pickup_delay_minutes
0,34,4.5,Jam,2,Snack,scooter,1,No,Metropolitian,33,Stormy,20.18,19,19,5
1,23,4.4,Low,0,Drinks,motorcycle,1,No,Urban,26,Sandstorms,1.55,8,8,15
2,32,4.0,Jam,0,Buffet,motorcycle,1,No,Metropolitian,47,Windy,13.97,21,21,15
3,26,4.9,Low,2,Buffet,scooter,0,No,Metropolitian,11,Stormy,3.11,8,8,15
4,22,4.8,Low,1,Snack,motorcycle,1,No,Metropolitian,28,Fog,10.87,23,23,10


### to-do 

In [ ]:
# 1 - encode 
# 2 - scaling 

# i think this it  g ! 

In [4]:
# encoding 
df.select_dtypes(include="object")

,Road_traffic_density,Type_of_order,Type_of_vehicle,Festival,City,weather_clean
0,Jam,Snack,scooter,No,Metropolitian,Stormy
1,Low,Drinks,motorcycle,No,Urban,Sandstorms
2,Jam,Buffet,motorcycle,No,Metropolitian,Windy
3,Low,Buffet,scooter,No,Metropolitian,Stormy
4,Low,Snack,motorcycle,No,Metropolitian,Fog
...,...,...,...,...,...,...
2873,Medium,Drinks,scooter,No,Metropolitian,Cloudy
2874,Jam,Meal,motorcycle,No,Metropolitian,Sunny
2875,Jam,Snack,motorcycle,No,Metropolitian,Sunny
2876,Jam,Snack,motorcycle,No,Metropolitian,Fog


In [3]:
y = df['delivery_time_mins']
X = df.drop(columns= ['delivery_time_mins'])

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size= 0.2,random_state=42)

In [48]:
X_train.shape

(2302, 14)

In [9]:
X_test.shape

(576, 14)

In [10]:
y_train.shape

(2302,)

In [11]:
y_test.shape

(576,)

#### Encoding after splitting the dataset to avoid data leakage

'''weather_clean
City
Type_of_vehicle
Type_of_order  we will do one-hot encoding cauz no order i mean no ranking'''

traffic_density ordinal encoding cauz it has categories  
'festival - binary encoding '

In [5]:
binary_cols = ["Festival"]

ordinal_cols = ["Road_traffic_density"]

nominal_cols = [
    "weather_clean",
    "City",
    "Type_of_order",
    "Type_of_vehicle"
]

In [50]:
for col in [
    "Festival",
    "Road_traffic_density",
    "weather_clean",
    "City",
    "Type_of_order",
    "Type_of_vehicle"]:
    print(col)
    print(df[col].unique())
    print()

Festival
['No' 'Yes']

Road_traffic_density
['Jam' 'Low' 'Medium' 'High']

weather_clean
['Stormy' 'Sandstorms' 'Windy' 'Fog' 'Cloudy' 'Sunny']

City
['Metropolitian' 'Urban' 'Semi-Urban']

Type_of_order
['Snack' 'Drinks' 'Buffet' 'Meal']

Type_of_vehicle
['scooter' 'motorcycle' 'electric_scooter']



In [6]:
# Binary Encoding
X_train["Festival"] = X_train["Festival"].map({"No": 0, "Yes": 1})
X_test["Festival"] = X_test["Festival"].map({"No": 0, "Yes": 1})

# Ordinal Encoding
traffic_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Jam": 3
}

X_train["Road_traffic_density"] = X_train["Road_traffic_density"].map(traffic_mapping)
X_test["Road_traffic_density"] = X_test["Road_traffic_density"].map(traffic_mapping)

In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train[nominal_cols])
X_test_encoded = encoder.transform(X_test[nominal_cols])

In [8]:
encoded_cols = encoder.get_feature_names_out(nominal_cols)

In [9]:
import pandas as pd

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_cols,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_cols,
    index=X_test.index
)

In [10]:
X_train = X_train.drop(columns=nominal_cols)
X_test = X_test.drop(columns=nominal_cols)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [15]:
print(X_train.dtypes)
print(y_test.dtypes)

Delivery_person_Age             int64
Delivery_person_Ratings       float64
Road_traffic_density            int64
Vehicle_condition               int64
multiple_deliveries             int64
Festival                        int64
distance_km                   float64
order_hour                      int64
pickup_hour                     int64
pickup_delay_minutes            int64
weather_clean_Fog             float64
weather_clean_Sandstorms      float64
weather_clean_Stormy          float64
weather_clean_Sunny           float64
weather_clean_Windy           float64
City_Semi-Urban               float64
City_Urban                    float64
Type_of_order_Drinks          float64
Type_of_order_Meal            float64
Type_of_order_Snack           float64
Type_of_vehicle_motorcycle    float64
Type_of_vehicle_scooter       float64
dtype: object
int64


In [18]:
X_test.to_csv("../data/processed/x_test.csv", index =False)

y_test.to_csv("../data/processed/y_test.csv", index =False)

X_train.to_csv("../data/processed/X_train.csv", index =False)

y_train.to_csv("../data/processed/y_train.csv", index =False)

In [ ]:
# we will scale based on the model that we are using !!!!